# pyFracAggregate — Comprehensive Demo

This notebook demonstrates the full capabilities of **pyFracAggregate**, a Python library for generating and analyzing fractal aggregates.

**Contents:**
1. [Introduction & Setup](#Introduction-&-Setup)
2. [Core Workflow — End-to-End](#Core-Workflow)
3. [Generation Methods — Comparing Four Algorithms](#Generation-Methods)
4. [Analysis & Visualization — Morphology and Fractal Dimension](#Analysis-&-Visualization)
5. [Export & Rendering — Format Details](#Export-&-Rendering)

In [ ]:
import pyFracAggregate as pfa
import numpy as np
import os

print(f"pyFracAggregate version: {pfa.__version__}")

# Check optional visualization dependencies
try:
    import pyvista
    print(f"pyvista {pyvista.__version__} — 3D rendering available")
except ImportError:
    print("pyvista not installed — 3D rendering (export_render, export_rotation_video) will not work")

try:
    import imageio
    print(f"imageio {imageio.__version__} — video export available")
except ImportError:
    print("imageio not installed — export_rotation_video will not work")

---

## Core Workflow — End-to-End

This section walks through the complete pipeline: **generate → analyze → export**. By the end, you'll have a full set of output files from a single PCA aggregate.

In [ ]:
# Generate a fractal aggregate using Particle-Cluster Aggregation (PCA)
agg = pfa.generate(
    n_particles=100,
    df=1.8,       # Fractal dimension
    kf=1.2,       # Fractal prefactor
    method='pca'
)

print(f"Particles: {agg.current_size}")
print(f"Radius range: [{agg.radii.min():.3f}, {agg.radii.max():.3f}]")
print(f"Position bounds: x[{agg.positions[:,0].min():.1f}, {agg.positions[:,0].max():.1f}]"
      f" y[{agg.positions[:,1].min():.1f}, {agg.positions[:,1].max():.1f}]"
      f" z[{agg.positions[:,2].min():.1f}, {agg.positions[:,2].max():.1f}]")

In [ ]:
# Analyze morphological properties
results = pfa.analyze(agg)

print("Analysis Results:")
print(f"  Number of particles (N): {results['N']}")
print(f"  Radius of gyration (Rg): {results['Rg']:.2f}")
print(f"  Center of mass (CoM):    [{results['CoM'][0]:.2f}, {results['CoM'][1]:.2f}, {results['CoM'][2]:.2f}]")
print(f"  Estimated Df:            {results['Df_estimated']:.2f} (input: 1.80)")
print(f"  Fit quality (R²):        {results['R2']:.4f}")

In [ ]:
os.makedirs("output", exist_ok=True)

# Export to YAML — full snapshot with metadata
pfa.export_yaml(
    agg, "output/aggregate.yaml",
    generation_params={"method": "pca", "n_particles": 100, "df": 1.8, "kf": 1.2},
    analysis_results=results,
)
print("\u2713 YAML exported")

# Export to VTK — point cloud
pfa.export_vtk(agg, "output/points.vtk")
print("\u2713 VTK point cloud exported")

# Export to VTM — MultiBlock (spheres as structured data)
pfa.export_vtm(agg, "output/blocks.vtm")
print("\u2713 VTM MultiBlock exported")

In [ ]:
# Render a 3D screenshot
pfa.export_render(agg, "output/aggregate_render.png", camera_position="iso")
print("\u2713 Rendered PNG saved")

# Generate a 360\u00b0 rotation video
pfa.export_rotation_video(agg, "output/aggregate_rotation.mp4", n_frames=72, fps=24)
print("\u2713 Rotation video saved")

print("\nAll exports complete. Check the 'output' directory.")

---

## Generation Methods — Comparing Four Algorithms

pyFracAggregate provides four generation algorithms:

| Method | Key | Description |
|--------|-----|-------------|
| **PCA** | `'pca'` | Particle-Cluster Aggregation — adds one particle at a time |
| **CCA** | `'cca'` | Cluster-Cluster Aggregation — merges sub-clusters |
| **FracVAL** | `'fracval'` | FracVAL algorithm — tunable fractal dimension |
| **TDCCA** | `'tdcca'` | Thouy & Jullien (2004) — cluster-cluster with strict fractal scaling |

In [ ]:
# Compare all four generation methods with the same fractal parameters
methods = ['pca', 'cca', 'fracval', 'tdcca']
aggregates = {}

for method in methods:
    print(f"Generating with {method.upper()}...")
    aggregates[method] = pfa.generate(
        n_particles=50,   # Small for speed
        df=1.8,
        kf=1.2,
        method=method,
    )

print("\n--- Comparison ---")
print(f"{'Method':<10} {'N':>5} {'Rg':>8}")
print("-" * 26)
for method, agg_m in aggregates.items():
    rg = pfa.radius_of_gyration(agg_m)
    print(f"{method:<10} {agg_m.current_size:>5} {rg:>8.2f}")

### Placement Strategies

Two placement strategies control how particles are positioned during generation:

- **`'algebraic'`** (default) — Analytical touching-point computation with Monte Carlo fallback. More accurate.
- **`'random'`** — Pure Monte Carlo sampling. Faster for large aggregates.

In [ ]:
# Compare placement strategies
for placement in ['algebraic', 'random']:
    agg_p = pfa.generate(n_particles=50, df=1.8, kf=1.2, method='pca', placement=placement)
    rg = pfa.radius_of_gyration(agg_p)
    print(f"placement={placement:<10} → N={agg_p.current_size}, Rg={rg:.2f}")

### Particle Distributions

By default, particles have uniform radius (`Monodisperse`). You can also use a `LognormalDistribution` for polydisperse systems:

In [ ]:
# Monodisperse vs Lognormal distribution
mono = pfa.Monodisperse(radius=1.0)
lognorm = pfa.LognormalDistribution(mean=1.0, std=0.3)

for name, dist in [("Monodisperse(1.0)", mono), ("Lognormal(1.0, 0.3)", lognorm)]:
    agg_d = pfa.generate(n_particles=50, df=1.8, kf=1.2, method='pca', particle_dist=dist)
    print(f"{name:<25} → N={agg_d.current_size}, Rg={pfa.radius_of_gyration(agg_d):.2f}")
    print(f"  Radii: min={agg_d.radii.min():.3f}, max={agg_d.radii.max():.3f}, mean={agg_d.radii.mean():.3f}")